In [ ]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from tools import plot_gtf
import re
import gc
from focal_regions import define_focal_regions, compute_focal_index, assign_region_to_locus, get_snp_array_counts
plt.rcParams['pdf.fonttype'] = 42

def natural_sort_key(s):
    """A natural sort key function."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

# Focal deletions

In [ ]:
df_all = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
df_age = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID_age.40709.txt', sep='\t')
gtf = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/GRCh38_supp_files/hg38.refGene.txt', sep='\t')
cyto = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/GRCh38_supp_files/cytoBand.txt.gz', sep='\t', header=None)
chip_genes = {'DLEU2', 'DNMT3A', 'CHEK2', 'TET2', 'NF1', 'ASXL1'}
fragile_genes = { # from https://www.nature.com/articles/s41586-019-1913-9/figures/15
    'FHIT', 'MACROD2', 'WWOX', 'IMMP2L', 'NAALADAL2', 'LRP1B', 'PDE4D', 
    'CCSER1', 'DMD', 'PRKN', 'SMYD3', 'PTPRD', 'LSAMP', 'AUTS2', 'RBFOX1', 
    'CSMD1', 'PRKG1', 'DIAPH2', 'NEGR1', 'GPC6', 'CTNNA3'
}
snp_array_hotspots = {'DLEU2', 'TET2', 'DNMT3A', '16p11.2', 'CHEK2', '11p15.5', 'NF1'}
df_snp_array = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.mCA_calls.snp_array.txt', sep='\t')[['CHR', 'START_MB', 'END_MB']]
breakpoints = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/breakpoints/WGS_500k.discordant_breakpoints.txt.gz', sep='\t')
breakpoints = breakpoints.query('orientation=="INNER" and type=="LOSS" or orientation=="OUTER" and type=="GAIN"').drop('type', axis=1).reset_index(drop=True)

In [ ]:
df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
df = define_focal_regions(df)
focal_index, focal_index_position = compute_focal_index(df)
df, gene_density = assign_region_to_locus(df, gtf, cyto, chip_genes, focal_index_position)
snp_array = get_snp_array_counts(df, df_snp_array)
df_breakpoints = df.merge(breakpoints, on = ['ID', 'chr', 'bpStart', 'bpEnd'], how = 'inner')
df_breakpoints = df_breakpoints[['ID', 'chr', 'leftBreakpoint', 'rightBreakpoint', 'type', 'region']] \
    .rename({'leftBreakpoint': 'bpStart', 'rightBreakpoint':'bpEnd'}, axis = 1)
df_breakpoints['length'] = df_breakpoints['bpEnd'] - df_breakpoints['bpStart']
gc.collect()

In [ ]:
breakpoint_by_cf = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t') \
    .query('type in ["LOSS", "GAIN"]') \
    .query('p!="T" and q!="T"') \
    .merge(breakpoints, how='outer')[['cf', 'potentialBreakpoints']].fillna(0) \
    .assign(has_breakpoints = lambda x: x['potentialBreakpoints']>0) \
    .assign(cf_bin = lambda x: pd.cut(x['cf'], bins=np.arange(0, 1.1, 0.1), right=False)) \
    .groupby('cf_bin') \
    .agg(
        has_breakpoints = ('has_breakpoints', 'sum'), 
        n = ('has_breakpoints', 'size'), 
        cf = ('cf', 'mean')) \
    .assign(no_breakpoints = lambda x: x['n'] - x['has_breakpoints']) 


import scipy.stats
lower = scipy.stats.beta(1/2+breakpoint_by_cf['has_breakpoints'], 1/2+breakpoint_by_cf['no_breakpoints']).ppf(0.025)
upper = scipy.stats.beta(1/2+breakpoint_by_cf['has_breakpoints'], 1/2+breakpoint_by_cf['no_breakpoints']).ppf(0.975)
fig, ax = plt.subplots(1, 1, figsize=(6,4), dpi=150)
ax.errorbar(
    breakpoint_by_cf['cf'], 
    breakpoint_by_cf['has_breakpoints']/breakpoint_by_cf['n'], 
    yerr = [breakpoint_by_cf['has_breakpoints']/breakpoint_by_cf['n'] - lower, 
            upper - breakpoint_by_cf['has_breakpoints']/breakpoint_by_cf['n']],
    marker='o',
    linestyle='none',
    capsize=5
)
ax.set_xlim([0, 1])
ax.set_xlabel('mCA cell fraction', fontsize=14)
ax.set_ylabel('Fraction of interstitial DEL/DUP mCAs\nwith discordant read support', fontsize=12)
ax.grid(axis='y')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.savefig('breakpoint_localization_by_cf.pdf', transparent=True, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6,6), dpi=150)
ax[0].hist(
    np.concatenate([
        np.abs(breakpoints['leftBreakpoint'] - breakpoints['bpStart']), 
        np.abs(breakpoints['rightBreakpoint'] - breakpoints['bpEnd'])
    ]),
    bins = np.linspace(0, 50000, 50),
    edgecolor='k'
)

ax[1].hist(
    np.abs((breakpoints['bpEnd'] - breakpoints['bpStart']) - (breakpoints['rightBreakpoint'] - breakpoints['leftBreakpoint'])),
    bins = np.linspace(0, 100000, 50),
    edgecolor='k'
)

ax[0].set_xlabel('Distance to nearest mCA breakpoint (bp)', fontsize=14)
ax[0].set_ylabel('Number of breakpoints', fontsize=14)
ax[0].text(-0.2, 1.1, 'a', transform = ax[0].transAxes, fontsize=18, ha='right')

ax[1].set_xlabel('Absolute discrepancy between mCA size \n determined by BAF vs discordant reads (bp)', fontsize=14)
ax[1].set_ylabel('Number of mCAs', fontsize=12)
ax[1].text(-0.2, 1.1, 'b', transform = ax[1].transAxes, fontsize=18, ha='right')
plt.tight_layout()
plt.savefig('breakpoint_accuracy.pdf', transparent=True, bbox_inches='tight')

In [ ]:
def create_focal_region_table(df, focal_index, snp_array, breakpoints, gene_density, fragile_genes, snp_array_hotspots):
    sex_bias = df.drop_duplicates(subset=['ID', 'region', 'type', 'locus'])\
        .groupby(['region', 'type', 'locus'])\
        .agg(
            num_males=('sex', lambda x: np.sum(x=='M')),
            num_females=('sex', lambda x: np.sum(x=='F')),
            sex_bias=('sex', lambda x: np.mean(x=="M")),
        )
    table = df.merge(breakpoints, on=['ID', 'chr', 'bpStart', 'bpEnd'], how='left')\
        .groupby(['region', 'type', 'locus']).agg(
        q0_length = ('length', lambda x: np.round(np.quantile(x, 0)/1e3)),
        q25_length = ('length', lambda x: np.round(np.quantile(x, 0.25)/1e3)),
        q50_length = ('length', lambda x: np.round(np.quantile(x, 0.5)/1e3)),
        q75_length = ('length', lambda x: np.round(np.quantile(x, 0.75)/1e3)),
        q100_length = ('length', lambda x: np.round(np.quantile(x, 1)/1e3)),
        count = ('ID', len),
        disc_reads=('potentialBreakpoints', lambda x: np.sum(x>=1)),
        mean_cf=('cf', np.mean),
        std_cf=('cf', np.std)
    ) \
        .reset_index() \
        .merge(sex_bias, on=['region', 'type', 'locus'], how='left') \
        .merge(snp_array, on='region', how='left') \
        .replace({np.nan:0}) \
        .infer_objects() \
        .merge(
            pd.DataFrame(
                [(region, cn, index) for (region, cn), index in focal_index.items()], 
                columns=['region', 'type', 'focal_index']
            ), 
                on=['region', 'type']
        ) \
        .merge(
            df.drop_duplicates(subset=['ID', 'region']).groupby(['region', 'type']).agg(
                mean_age=('age', np.mean), 
                stderr_age=('age', lambda x: np.std(x)/np.sqrt(len(x)))
            ), 
            on = ['region', 'type']
        ) \
        .merge(
            pd.DataFrame(
                [(region, cn, len([k for k,v in genes.items() if v>0.5])) for (region, cn), genes in gene_density.items()],
                columns=['region', 'type', 'num_genes']
            ),
            on = ['region', 'type']
        ) \
        .assign(
            chrom=lambda x: [region.split(':')[0] for region in x['region']], 
            startMb=lambda x: [int(region.split(':')[1].split('-')[0])//1000000 for region in x['region']],
            endMb=lambda x: [int(region.split(':')[1].split('-')[1])//1000000 for region in x['region']]) \
        .astype({
            'q0_length':np.int64,
            'q25_length':np.int64,
            'q50_length':np.int64,
            'q75_length':np.int64,
            'q100_length':np.int64,
            'snp_array_count':np.int64,
            'region': str
        })  
    
    table['fragile'] = table['locus'].apply(lambda x: True if x in fragile_genes else False)
    table['snp_array_hotspot'] = table['locus'].apply(lambda x: True if x in snp_array_hotspots else False)

    table = table[['chrom', 'startMb', 'endMb'] + [x for x in table.columns if x not in {'chrom', 'startMb', 'endMb'}]]
    table = table.sort_values('count', ascending=False)
    return table


In [ ]:
table = create_focal_region_table(df, focal_index, snp_array, breakpoints, gene_density, fragile_genes, snp_array_hotspots)
table[
    [
        'chrom', 'startMb', 'endMb', 'type', 
        'locus', 'fragile', 'snp_array_hotspot', 
        'count', 'disc_reads', 
        'focal_index', 'mean_age', 'stderr_age', 'mean_cf', 'std_cf', 'sex_bias', 
        'q0_length', 'q25_length', 'q50_length', 'q75_length', 'q100_length'
    ]
].to_csv('focal_region_table.csv', sep=',', index=False)

In [ ]:
import scipy.stats
df_plot = table.query('type=="LOSS"')
alpha = 1
fig = plt.figure(figsize=(3*4,3*3), dpi=150)
ax = fig.add_axes([0.1, 0.2, 0.6, 0.7]) # [left, bottom, width, height]
cartoon_ax = fig.add_axes([0.1, 0.05, 0.6, 0.05])
for spine in ['left', 'right', 'top', 'bottom']:
    cartoon_ax.spines[spine].set_visible(False)
legend_ax = fig.add_axes([0.7, 0.2, 0.3, 0.6])
legend_ax.axis('off') 
cartoon_ax.set_xticks([])
cartoon_ax.set_yticks([])

xs = np.array([
    [0.85, 0.93],
    [0.87, 0.92],
    [0.86, 0.91],
    [0.88, 0.92],
    [0.885, 0.915],
    [0.89, 0.91],
    [0.887, 0.902],
    [0.89, 0.9],
    [0.89, 0.895],
    [0.897, 0.9],
]    
)
cartoon_ax.hlines(np.arange(1, 10+1), xs[:, 0], xs[:, 1], color='blue')

xs = np.array([
    [0.32, 0.36],
    [0.35, 0.40],
    [0.36, 0.41],
    [0.39, 0.44],
    [0.41, 0.45],
    [0.42, 0.44],
    [0.4, 0.41],
    [0.43, 0.435],
    [0.41, 0.415],
    [0.44, 0.443],
]    
)
cartoon_ax.hlines(np.arange(1, 10+1), xs[:, 0], xs[:, 1], color='blue')
cartoon_ax.annotate('', (0.8, 5), (0.5, 5), arrowprops=dict(arrowstyle="-|>", facecolor='k'),size=30)


texts = []
for _,row in df_plot.iterrows():
    if row['count'] < 15: continue
    if row['locus'] in {'RFC1', '10q25.2'}: continue
    color = 'mediumseagreen' if row['locus'] in chip_genes else 'orchid' if row['locus'] in fragile_genes else 'silver'
    fillstyle = 'none' if row['locus'] in snp_array_hotspots else 'full'
    ax.errorbar(
        row['focal_index'],
        row['mean_age'], 
        yerr=row['stderr_age']*1.96,
        capsize=0,
        marker='o',
        markersize=np.sqrt(row['count'])*2,
        c=color,
        linewidth=0.5,
        alpha = alpha,
        fillstyle=fillstyle
    )
    horizontal_alignment='left' if row['locus'] in {'TET2', 'CHEK2', 'NRXN1', 'PRKN', 'MACROD2', '16p11.2'} else 'right'
    horizontal_adjust = 0.015 if horizontal_alignment == 'left' else -0.015
    vertical_adjust = 0.2 if row['locus'] in {'11p15.5'} else 0
    if row['count'] < 25: continue
    ax.text(
        row['focal_index']+horizontal_adjust, 
        row['mean_age']+vertical_adjust, 
        row['locus'], 
        horizontalalignment=horizontal_alignment,
        fontstyle='normal' if row['locus'] in {'16p11.2', '11p15.5'} else 'italic',
        fontsize=18,
        color=color if color!='silver' else 'gray'
    )
    
ax.set_xlim(0.25, 1.05)
ax.set_xlabel('Focal index of deletion hotspot', fontsize=24)
ax.set_ylabel('Mean age (years)', fontsize=24)
ax.axhline(df_age['age'].mean(), color='k', ls=':')
cartoon_ax.set_xlim(ax.get_xlim())
ax.set_xticks(np.arange(0.3, 1+0.1, 0.1))
ax.tick_params(axis='both', which='major', labelsize=18)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

# Adding the legend as a separate axis
size1 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(20)*2, label = '20 mCAs', color='k')
size2 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(100)*2, label = '100 mCAs', color='k')
size3 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(500)*2, label = '500 mCAs', color='k')
size_legend = legend_ax.legend(
    handles=[size1,size2,size3], 
    loc='lower left', 
    frameon=False, 
    handletextpad=1, 
    labelspacing=1.5, 
    ncol=1, 
    bbox_to_anchor=(0, 0),
    fontsize=18
)

legend_ax.add_artist(size_legend)

color1 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(20)*2, label = 'CH driver', color='mediumseagreen')
color2 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(20)*2, label = 'Fragile site', color='orchid')
color3 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(20)*2, label = 'Other', color='silver')
color_legend = legend_ax.legend(
    handles=[color1, color2, color3], 
    loc='center left', 
    frameon=False, 
    handletextpad=1, 
    labelspacing=1, 
    ncol=1, 
    bbox_to_anchor=(0, 1),
    fontsize=18
)
legend_ax.add_artist(color_legend)

fill1 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(20)*2, label = 'Previously seen', color='k', fillstyle='none')
fill2 = legend_ax.errorbar([],[], linestyle='none', marker='o', markersize=np.sqrt(20)*2, label = 'New from WGS', color='k')
fill_legend = legend_ax.legend(
    handles=[fill1, fill2], 
    loc='upper left', 
    frameon=False, 
    handletextpad=1, 
    labelspacing=1, 
    ncol=1, 
    bbox_to_anchor=(0, 0.7),
    fontsize=18
)
legend_ax.add_artist(fill_legend)

plt.savefig('focal_event_overview.pdf', transparent=True, bbox_inches='tight')
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for i,(_,row) in enumerate(table.query('type=="LOSS"').sort_values('sex_bias').iterrows()):
    lower = scipy.stats.beta(row['num_males']+1/2, row['num_females']+1/2).ppf(0.025)
    upper = scipy.stats.beta(row['num_males']+1/2, row['num_females']+1/2).ppf(0.975)
    sig = (lower - 0.457) * (upper - 0.457) > 0
    bonferroni = scipy.stats.binomtest(row['num_males'], row['num_males']+row['num_females'], p=0.457).pvalue < 0.05/len(table.query('type=="LOSS"'))
    color = 'mediumseagreen' if row['locus'] in chip_genes else 'orchid' if row['locus'] in fragile_genes else 'gray'
    ax.errorbar(i, row['sex_bias'], yerr=[[row['sex_bias'] - lower], [upper - row['sex_bias']]], color=color, marker='o', alpha=0.1 if not sig else 1)
    fontstyle='normal' if 'p' in row['locus'] or 'q' in row['locus'] else 'italic'
    ax.text(i, row['sex_bias']+0.02, row['locus'], rotation=90, va='bottom', ha='right', color=color, fontstyle=fontstyle, alpha=0.5 if not sig else 1)
    p = scipy.stats.binomtest(row['num_males'], row['num_males']+row['num_females'], p=0.457).pvalue
    if sig: ax.text(i, 0.01, f'p={p:.2e}', ha='center', fontsize=8, rotation=90)
    # if bonferroni: ax.text(i, 0.1, '**', ha='center')
    # elif sig: ax.text(i, 0.1, '*', ha='center')
ax.axhline(0.457, color='k', ls=':')
ax.set_ylabel('Fraction male', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_xticks([])
for spine in ['right', 'top', 'bottom']:
    ax.spines[spine].set_visible(False)
plt.savefig('fragile_site_sex_bias.pdf', transparent=True, bbox_inches='tight')
plt.show()
# plt.close()

In [ ]:
df[['ID', 'chr', 'bpStart', 'bpEnd', 'depth', 'type', 'region', 'locus']].to_csv('~/out_dir/focal_events.txt', index=None, sep='\t')

In [ ]:
def plot_focal_pileup(df_chr, region, pad=0.5e6, legend=False, text=True):
    chrom = region.split(':')[0]
    start = int(max(natural_sort_key(region)[3] - pad, 0))
    end = int(natural_sort_key(region)[5] + pad)

    counts = defaultdict(int)
    for idx, row in df_chr.iterrows():
        for i in range(start, end+1, 1000):
            if i > row['bpStart'] and i < row['bpEnd']:
                counts[i] += 1
    counts = np.array([[k, v] for k,v in sorted(counts.items(), key=lambda x: x[0])])


    buffer = np.array([1 if x =='GAIN' else 2 if x=='CN-LOH' else 3 for x in df_chr['type']])
    ycoord = np.argsort(np.argsort(df_chr['bpStart'] - df_chr['length'] + 1e9 * buffer))

    color = ['r' if x=='GAIN' else 'gold' if x=='CN-LOH' else 'b' for x in df_chr['type']]


    # fig = plt.figure(dpi=150, figsize=((end-start)/1e6, len(ycoord)/20))
    fig = plt.figure(dpi=150, figsize=(6, 7))

    ax2 = fig.add_axes([0.1, 0.1, 0.9, 0.2])
    ax1 = fig.add_axes([0.1, 0.4, 0.9, 0.5])
    ax3 = fig.add_axes([0.1, 0.3, 0.9, 0.1])

    N=50
    ax3.plot(counts[:, 0]/1e6, np.convolve(counts[:,1], np.ones(N)/N, mode = 'same'), color='k', linewidth = 0.5)
    ax3.fill_between(
        counts[:, 0]/1e6, 
        np.convolve(counts[:,1], np.ones(N)/N, mode = 'same'), 
        color='gray',
        alpha = 0.5)

    for ax in (ax1, ax2, ax3):
        ax.set_yticks([])
        ax.set_xlim(start/1e6, end/1e6)
        ax.spines['top'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if ax == ax2: continue
        ax.set_xticks([])
        ax.spines['bottom'].set_visible(False)

    ax1.hlines(
        ycoord,
        df_chr['bpStart']/1e6,
        df_chr['bpEnd']/1e6,
        color = color
    )
    ax1.set_ylim(-1, 100)

    # plot_gtf(gtf, chrom, start, end, ax2, highlight_genes={'MACROD2', 'RBFOX1', 'NRXN1', 'SETMAR', 'IMMP2L', 'DLEU2', 'DNMT3A', 'PRKN'}
    plot_gtf(gtf.query('coding_gene'), chrom, start, end, ax2, names=text)
    ax2.set_ylim(-4, 6)
    xticks = np.arange(int(start/1e6*4)/4, int((end/1e6)*4)/4 + 0.01, 0.25)
    xticklabels = [int(tick) if tick.is_integer() else '' for tick in xticks]
    ax2.set_xticks(xticks, xticklabels, fontsize=16)
    ax3.text(ax3.get_xlim()[0], ax3.get_ylim()[1]/2, 'mCA coverage', fontsize=16)

    ax2.set_xlabel(f'Chromosome {chrom[3:]} position (Mb)', fontsize=18)
    if legend:
        ax1.plot([],[],color='r',label='Gain')
        ax1.plot([],[],color='b',label='Loss')
        ax1.legend(loc='center left', frameon=False, title='mCA', fontsize=18, title_fontsize=18)

In [ ]:
for locus in ['RBFOX1', 'MACROD2', 'PRKN', 'NRXN1']:
    region=table.query('locus==@locus')['region'].to_numpy()[0]
    chrom = region.split(':')[0]
    start = natural_sort_key(region)[3]
    end = natural_sort_key(region)[5]

    df_chr = df_breakpoints.query('region==@region')

    pad = 0.5e6
    legend = False
    if locus == 'MACROD2':
        pad = 0.1e6
    if locus == 'RBFOX1':
        legend = True

    plot_focal_pileup(df_chr, region, pad = pad, legend=legend)
    plt.savefig(f'{locus}.pdf', bbox_inches='tight', transparent=True)
    plt.close()


# Microhomology

In [ ]:
split_reads = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/breakpoints/WGS_500k.split_breakpoints.txt.gz', sep='\t')
split_reads = split_reads.query('potentialBreakpoints==1 and orientation=="INNER"')
# split_reads = split_reads.iloc[:, [0, 1, 2, 6, 12]]
# track instances where a given deletion has more than 1 exact breakpoint suggested by split reads
# split_reads = split_reads.merge(split_reads.value_counts(subset=['ID', 'region']).reset_index(), on=['ID', 'region'])

def draw_brace(ax, xspan, yy, text, fontsize):
    """Draws an annotated brace on the axes."""
    xmin, xmax = xspan
    xspan = xmax - xmin
    ax_xmin, ax_xmax = ax.get_xlim()
    xax_span = ax_xmax - ax_xmin

    ymin, ymax = ax.get_ylim()
    yspan = ymax - ymin
    resolution = int(xspan/xax_span*100)*2+1 # guaranteed uneven
    beta = 300./xax_span # the higher this is, the smaller the radius

    x = np.linspace(xmin, xmax, resolution)
    x_half = x[:int(resolution/2)+1]
    y_half_brace = (1/(1.+np.exp(-beta*(x_half-x_half[0])))
                    + 1/(1.+np.exp(-beta*(x_half-x_half[-1]))))
    y = np.concatenate((y_half_brace, y_half_brace[-2::-1]))
    y = yy + (.05*y - .01)*yspan # adjust vertical position

    ax.autoscale(False)
    ax.plot(x, y, color='black', lw=1)

    ax.text((xmax+xmin)/2., yy+.07*yspan, text, ha='center', va='bottom', fontsize=fontsize)

microhomology = split_reads['microhomology'].value_counts().reset_index().to_numpy()
fig, ax = plt.subplots(dpi=150)
ax.bar(microhomology[:,0], microhomology[:,1], width=0.8, edgecolor='k')
ax.set_xlabel('Microhomology (base pairs)', fontsize=18)
ax.set_ylabel('Number of mCA breakpoints', fontsize=18)
ax.set_xticks(np.arange(0, 50, 5))
ax.set_xlim(-1, 20)
ax.set_ylim(0, 900)
ax.tick_params(labelsize=16)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

draw_brace(ax, (0, 4), 800, 'NHEJ',fontsize=16)
draw_brace(ax, (2, 20), 750, 'Homology Directed Repair',fontsize=16)
plt.savefig('microhomology.pdf',transparent=True,bbox_inches='tight')
plt.close()

# Complex rearrangements

In [ ]:
def plot_chrom(chrom, ax, y_coord=0):
    cyto = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/GRCh38_supp_files/cytoBand.txt.gz', sep ='\t', header=None)
    cyto.columns = ['chrom', 'start', 'stop', 'label', 'stain']
    cyto['start'] = cyto['start']/1e6
    cyto['stop'] = cyto['stop']/1e6
    cyto['width'] = cyto['stop'] - cyto['start']

    color_lookup = {
        'gneg': (1., 1., 1.),
        'gpos25': (.6, .6, .6),
        'gpos50': (.4, .4, .4),
        'gpos75': (.2, .2, .2),
        'gpos100': (0., 0., 0.),
        'acen': (.8, .4, .4),
        'gvar': (.8, .8, .8),
        'stalk': (.9, .9, .9)
    }
    cyto_chr = cyto.query('chrom==@chrom')

    ax.broken_barh(
        cyto_chr.loc[:, ['start', 'width']].to_numpy(), 
        [y_coord, 20], 
        facecolor = [color_lookup[x] for x in cyto_chr['stain']],
        edgecolor = 'black'
    )

def draw_complex_sv(chrom, xs, gaps, y_line, y_chrom, ax, color, below=False):
    plot_chrom(chrom, ax, y_chrom)
    ys = np.repeat(y_line, len(xs)/2)
    starts = xs[::2]
    ends = xs[1::2]
    ax.hlines(ys, starts, ends, linewidth=20, capstyle='round', colors=color)
    for x0, x1 in zip(gaps, xs[1:-1]):
        ax.plot([x0, x1], [y_chrom+(0 if below else 20), y_line+(5 if below else -5)], color='k')

    ax.set_ylim((-8.5, 178.5))
    ax.set_xlim((-12.109676450000002, 254.30320545))


In [ ]:
fig, ax = plt.subplots(dpi=150, figsize = (24, 8))

for loc in ['left', 'right', 'top', 'bottom']:
    ax.spines[loc].set_visible(False)

ax.set_xticks([])
ax.set_yticks([])

c1 = 'lightsalmon'
c2 = 'lightgreen'


xs = np.array([0, 20, 30, 65, 75, 90, 100, 240])
gaps = np.array([24, 25, 69.9, 70.1, 85, 86])
draw_complex_sv('chr2', xs, gaps, 50, 0, ax, c1)
ax.plot(110, 50, marker='o', markersize=30, color='k')
ax.annotate('DNMT3A', (25.2, 0), (25.2, -24), fontsize=32, fontstyle='italic')
ax.plot([25.2, 25.2], [-10, -1], color='k', linewidth=3)
ax.text(-5, 0, 'chr2', fontsize=48, horizontalalignment='right')
ax.text(-5, 150, 'chr4', fontsize=48, horizontalalignment='right')

xs = np.array([0, 20, 30, 190])
gaps = np.array([27, 29])
draw_complex_sv('chr4', xs, gaps, 120, 150, ax, c2, True)
ax.plot(50, 120, marker='o', markersize=30, color='k')

plt.savefig('complex_sv_cartoon.1.pdf', transparent=True, bbox_inches='tight')
plt.close()

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize = (24, 8))

for loc in ['left', 'right', 'top', 'bottom']:
    ax.spines[loc].set_visible(False)

ax.set_xticks([])
ax.set_yticks([])

c1 = 'slategray'

xs = np.array([0, 50, 60, 73, 80, 91, 97, 107])
gaps = np.array([74, 78, 79, 80, 83, 84])
draw_complex_sv('chr14', xs, gaps, 50, 0, ax, 'slategray')
ax.plot(10, 50, marker='o', markersize=30, color='k')
ax.text(-5, 0, 'chr14', fontsize=48, horizontalalignment='right')

plt.savefig('complex_sv_cartoon.2.pdf', transparent=True, bbox_inches='tight')
plt.close()